# NB01 — Data Exploration & Audit

**Purpose:** Understand the raw data before computing any metrics.
- Load all 26 × 13 DICOMs
- Inspect DICOM tags (image dimensions, pixel spacing, frequency offset)
- Load & display the Ground Truth Excel
- Visualise all 13 offset images for sample patients, GT highlighted
- Flag Patient-KP (7 images, apical slice anomaly)

**Runs on:** Local CPU — no GPU needed.

## 0. Install Dependencies

In [9]:
# Install required libraries (safe to re-run — already-installed packages are skipped)
import subprocess, sys

packages = [
    'pydicom',     # DICOM reading
    'numpy',       # numerical arrays
    'pandas',      # dataframes / Excel
    'matplotlib',  # plotting
    'Pillow',      # image resizing (PIL)
    'openpyxl',   # .xlsx reading backend
    'scipy',       # image filters
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('All dependencies installed / already present.')

All dependencies installed / already present.


In [10]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0f0f1a'
matplotlib.rcParams['axes.facecolor']   = '#1a1a2e'
matplotlib.rcParams['text.color']       = 'white'
matplotlib.rcParams['axes.labelcolor']  = 'white'
matplotlib.rcParams['xtick.color']      = 'gray'
matplotlib.rcParams['ytick.color']      = 'gray'
%matplotlib inline

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_ROOT = r'E:/CINE-Test1/Dr-Dataset/1st_cohort_SAX'
GT_PATH   = r'E:/CINE-Test1/1stcohortSAX (1).xlsx'
OUT_DIR   = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print('Paths OK')

Paths OK


## 1. Load Ground Truth

In [11]:
from utils.gt_loader import load_gt, gt_as_dataframe

gt_dict = load_gt(GT_PATH)
gt_df   = gt_as_dataframe(gt_dict)

print(f'Loaded GT for {len(gt_df)} patients')
print(gt_df.to_string(index=False))

Loaded GT for 26 patients
  patient_id  image_no  offset_hz  manual_hz  n_images
 Patient-BEV         7        0.0        0.0        13
 Patient-BGT         7        0.0        0.0        13
 Patient-BVE         6      -25.0       25.0        13
 Patient-CEA         7        0.0        0.0        13
Patient-CEJP         4      -75.0       75.0        13
 Patient-DEM         9       50.0      -50.0        13
 Patient-FAJ         9       50.0      -50.0        13
 Patient-FEC         8       25.0      -25.0        13
  Patient-GJ         9       50.0      -50.0        13
 Patient-HMJ         8       25.0      -25.0        13
 Patient-HRJ         7        0.0        0.0        13
 Patient-IAM         8       25.0      -25.0        13
 Patient-INA        11      100.0     -100.0        13
  Patient-KP         6      100.0     -100.0         7
  Patient-MJ         7        0.0        0.0        13
  Patient-MS         9       50.0      -50.0        13
 Patient-MSR        10       75.0      

## 2. Load All Patient DICOMs

In [12]:
from utils.dicom_loader import load_all_patients

all_data = load_all_patients(DATA_ROOT, normalise=True, verbose=True)
print(f'\nTotal patients loaded: {len(all_data)}')

  Loaded Patient-BEV: 13 images, step=25 Hz
  Loaded Patient-BGT: 13 images, step=25 Hz
  Loaded Patient-BVE: 12 images, step=27 Hz
  Loaded Patient-CEA: 13 images, step=25 Hz
  Loaded Patient-CEJP: 13 images, step=25 Hz
  Loaded Patient-DEM: 13 images, step=25 Hz
  Loaded Patient-FAJ: 13 images, step=25 Hz
  Loaded Patient-FEC: 13 images, step=25 Hz
  Loaded Patient-GJ: 13 images, step=25 Hz
  Loaded Patient-HMJ: 13 images, step=25 Hz
  Loaded Patient-HRJ: 13 images, step=25 Hz
  Loaded Patient-IAM: 13 images, step=25 Hz
  Loaded Patient-INA: 13 images, step=25 Hz
  Loaded Patient-KP: 7 images, step=50 Hz
  Loaded Patient-MJ: 13 images, step=25 Hz
  Loaded Patient-MS: 13 images, step=25 Hz
  Loaded Patient-MSR: 13 images, step=25 Hz
  Loaded Patient-OAF: 13 images, step=25 Hz
  Loaded Patient-PJ: 13 images, step=25 Hz
  Loaded Patient-RAK: 13 images, step=25 Hz
  Loaded Patient-RK: 13 images, step=25 Hz
  Loaded Patient-SK: 13 images, step=25 Hz
  Loaded Patient-SWJ: 13 images, step=2

## 3. Dataset Summary Table

In [13]:
import pydicom
from pathlib import Path
import glob

summary_rows = []
for pid, data in sorted(all_data.items()):
    imgs = data['images']
    offsets = data['offsets_hz']
    step = data['step_hz']
    H, W = imgs[0].shape[:2]

    dcm = data['dcm_list'][0] if data['dcm_list'] else None
    px_space = tuple(float(x) for x in dcm.PixelSpacing) if (dcm and hasattr(dcm,'PixelSpacing')) else ('?','?')

    gt_no  = gt_dict[pid]['image_no'] if pid in gt_dict else '?'
    gt_hz  = gt_dict[pid]['offset_hz'] if pid in gt_dict else '?'
    flag   = '⚠ KP (atypical)' if pid == 'Patient-KP' else (
             '⚠ BVE (12 imgs)' if len(imgs) != 13 and pid != 'Patient-KP' else '')

    summary_rows.append({
        'Patient':     pid,
        'N_images':    len(imgs),
        'Step_Hz':     step,
        'H x W':       f'{H}x{W}',
        'PixelSpacing': f'{px_space[0]:.2f}x{px_space[1]:.2f}' if px_space[0]!='?' else '?',
        'GT_image_no': gt_no,
        'GT_Hz':       gt_hz,
        'Flag':        flag,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(f'{OUT_DIR}/dataset_summary.csv', index=False)
print(f'\nSaved: {OUT_DIR}/dataset_summary.csv')

     Patient  N_images   Step_Hz   H x W PixelSpacing  GT_image_no  GT_Hz            Flag
 Patient-BEV        13 25.000000 256x208    1.33x1.33            7    0.0                
 Patient-BGT        13 25.000000 256x208    1.33x1.33            7    0.0                
 Patient-BVE        12 27.272727 256x208    1.33x1.33            6  -25.0 ⚠ BVE (12 imgs)
 Patient-CEA        13 25.000000 256x230    1.56x1.56            7    0.0                
Patient-CEJP        13 25.000000 256x208    1.33x1.33            4  -75.0                
 Patient-DEM        13 25.000000 256x208    1.33x1.33            9   50.0                
 Patient-FAJ        13 25.000000 256x208    1.41x1.41            9   50.0                
 Patient-FEC        13 25.000000 208x256    1.33x1.33            8   25.0                
  Patient-GJ        13 25.000000 256x208    1.48x1.48            9   50.0                
 Patient-HMJ        13 25.000000 208x256    1.48x1.48            8   25.0                
 Patient-H

## 4. DICOM Tag Inspection (Sample Patient)

In [14]:
SAMPLE_PID = 'Patient-BEV'
dcm = all_data[SAMPLE_PID]['dcm_list'][6]   # middle image

tags_of_interest = [
    'PatientID', 'StudyDescription', 'SeriesDescription',
    'Modality', 'Rows', 'Columns', 'PixelSpacing',
    'NumberOfFrames', 'BitsAllocated', 'BitsStored',
    'SliceThickness',
]

print(f'=== DICOM Tags for {SAMPLE_PID} image #7 ===')
for tag in tags_of_interest:
    val = getattr(dcm, tag, 'N/A')
    print(f'  {tag:30s}: {val}')

frames = getattr(dcm, 'NumberOfFrames', 1)
print(f'\n  Pixel array shape: {dcm.pixel_array.shape}')
print(f'  → Single-frame DICOM confirmed (1 cardiac phase per file, 13 files = 13 offsets)')

=== DICOM Tags for Patient-BEV image #7 ===
  PatientID                     : 
  StudyDescription              : 
  SeriesDescription             : 
  Modality                      : MR
  Rows                          : 256
  Columns                       : 208
  PixelSpacing                  : [1.32812, 1.32812]
  NumberOfFrames                : N/A
  BitsAllocated                 : 16
  BitsStored                    : 12
  SliceThickness                : 6

  Pixel array shape: (256, 208)
  → Single-frame DICOM confirmed (1 cardiac phase per file, 13 files = 13 offsets)


## 5. Visual Strip — Sample Patients (GT highlighted in green)

In [15]:
import matplotlib.pyplot as plt
from utils.vis_utils import plot_patient_strip

SHOW_PATIENTS = ['Patient-BEV', 'Patient-BGT', 'Patient-INA', 'Patient-KP']

for pid in SHOW_PATIENTS:
    if pid not in all_data:
        print(f'{pid} not found, skipping')
        continue
    data = all_data[pid]
    gt_no = gt_dict[pid]['image_no'] if pid in gt_dict else None
    fig = plot_patient_strip(
        images=data['images'],
        offsets_hz=data['offsets_hz'],
        patient_id=pid,
        gt_image_no=gt_no,
        save_path=f'{OUT_DIR}/strip_{pid}.png'
    )
    plt.show()
    print(f'Saved strip for {pid}')

C:\Users\Hp\AppData\Local\Temp\ipykernel_5980\1506454833.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved strip for Patient-BEV
Saved strip for Patient-BGT
Saved strip for Patient-INA
Saved strip for Patient-KP


## 6. GT Distribution Analysis

In [16]:
import matplotlib.pyplot as plt

gt_offsets    = [info['offset_hz'] for pid, info in gt_dict.items() if pid != 'Patient-KP']
gt_image_nos  = [info['image_no']  for pid, info in gt_dict.items() if pid != 'Patient-KP']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(gt_offsets, bins=13, color='#4fc3f7', edgecolor='#0f0f1a', alpha=0.85)
axes[0].axvline(0, color='#69f0ae', linestyle='--', label='0 Hz')
axes[0].set_xlabel('GT Offset (Hz)')
axes[0].set_ylabel('Count')
axes[0].set_title('GT Frequency Distribution (25 patients, KP excluded)')
axes[0].legend()

axes[1].hist(gt_image_nos, bins=range(1, 15), color='#ce93d8', edgecolor='#0f0f1a', alpha=0.85)
axes[1].axvline(7, color='#69f0ae', linestyle='--', label='#7 = 0 Hz')
axes[1].set_xlabel('GT image_no (1-indexed)')
axes[1].set_ylabel('Count')
axes[1].set_title('GT Image Number Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/gt_distribution.png', bbox_inches='tight', dpi=120)
plt.show()

print(f'GT offsets — mean: {np.mean(gt_offsets):.1f} Hz,  median: {np.median(gt_offsets):.1f} Hz')
print(f'GT image_no — min: {min(gt_image_nos)}, max: {max(gt_image_nos)}')

GT offsets — mean: 24.0 Hz,  median: 25.0 Hz
GT image_no — min: 4, max: 11


C:\Users\Hp\AppData\Local\Temp\ipykernel_5980\2042179528.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Pixel Intensity Statistics

In [17]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.set_facecolor('#1a1a2e')

for pid, data in sorted(all_data.items()):
    if pid == 'Patient-KP': continue
    means = [img.mean() for img in data['images']]
    ax.plot(data['offsets_hz'], means, alpha=0.4, linewidth=0.8, color='#4fc3f7')

ax.set_xlabel('Offset (Hz)')
ax.set_ylabel('Mean Pixel Intensity')
ax.set_title('Whole-Image Mean Intensity vs Frequency Offset (all patients)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/intensity_vs_offset.png', bbox_inches='tight', dpi=120)
plt.show()
print('Done. Proceed to nb02_segmentation_qa.ipynb')

Done. Proceed to nb02_segmentation_qa.ipynb


C:\Users\Hp\AppData\Local\Temp\ipykernel_5980\3140656014.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
